# Speech Emotion Recognition — RAVDESS Mel-Spectrogram + 2D CNN

Clean notebook for the new version.

**Pipeline**
- Actor-based train/validation/test split
- Augmentation only for training data
- Mel-spectrogram input
- 2D CNN
- One label encoder for train/validation/test
- Test set used only after training

In [ ]:
import os
import random
import pickle
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import librosa
import librosa.display

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dropout, Flatten, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Librosa:", librosa.__version__)

In [ ]:
RAVDESS_DIR = r"E:\Occupation\Machine Learning\CodeAlpha\Emotion Recognition from Speech\emotional-speech-audios\audio_speech_actors_01-24"

print("Dataset exists:", os.path.exists(RAVDESS_DIR))

if not os.path.exists(RAVDESS_DIR):
    raise FileNotFoundError("Please correct RAVDESS_DIR.")

In [ ]:
emotion_map = {
    1: "neutral",
    2: "calm",
    3: "happy",
    4: "sad",
    5: "angry",
    6: "fear",
    7: "disgust",
    8: "surprise"
}

records = []

for actor_folder in sorted(os.listdir(RAVDESS_DIR)):
    actor_path = os.path.join(RAVDESS_DIR, actor_folder)

    if not os.path.isdir(actor_path):
        continue

    for file in sorted(os.listdir(actor_path)):
        if not file.lower().endswith(".wav"):
            continue

        parts = file.split(".")[0].split("-")
        emotion_code = int(parts[2])
        actor_id = int(parts[6])

        records.append({
            "Emotions": emotion_map[emotion_code],
            "Path": os.path.join(actor_path, file),
            "Actor": actor_id
        })

df = pd.DataFrame(records)

print(df.head())
print("Total samples:", len(df))
print(df["Emotions"].value_counts())
print("Actors:", sorted(df["Actor"].unique()))

In [ ]:
REMOVE_CALM = False

if REMOVE_CALM:
    df = df[df["Emotions"] != "calm"].reset_index(drop=True)

num_classes = df["Emotions"].nunique()

print("Number of classes:", num_classes)
print(df["Emotions"].value_counts())

In [ ]:
train_actors = list(range(1, 17))   # 1-16
val_actors = list(range(17, 19))    # 17-18
test_actors = list(range(19, 25))   # 19-24

train_df = df[df["Actor"].isin(train_actors)].reset_index(drop=True)
val_df = df[df["Actor"].isin(val_actors)].reset_index(drop=True)
test_df = df[df["Actor"].isin(test_actors)].reset_index(drop=True)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

print("\nTrain actors:", sorted(train_df["Actor"].unique()))
print("Validation actors:", sorted(val_df["Actor"].unique()))
print("Test actors:", sorted(test_df["Actor"].unique()))

print("\nTrain emotion counts:")
print(train_df["Emotions"].value_counts())

print("\nValidation emotion counts:")
print(val_df["Emotions"].value_counts())

print("\nTest emotion counts:")
print(test_df["Emotions"].value_counts())

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, y="Emotions", order=df["Emotions"].value_counts().index)
plt.title("Emotion Count")
plt.show()

In [ ]:
def add_noise(data, noise_factor=0.035):
    noise_amp = noise_factor * np.random.uniform() * np.max(np.abs(data))
    return data + noise_amp * np.random.normal(size=data.shape[0])

def time_stretch(data, rate=0.85):
    return librosa.effects.time_stretch(y=data, rate=rate)

def pitch_shift(data, sr, n_steps=0.7):
    return librosa.effects.pitch_shift(y=data, sr=sr, n_steps=n_steps)

In [ ]:
SAMPLE_RATE = 22050
DURATION = 3.0
OFFSET = 0.5
N_MELS = 128
MAX_LEN = 130

def load_audio(path):
    data, sr = librosa.load(path, sr=SAMPLE_RATE, duration=DURATION, offset=OFFSET)
    return data, sr

def audio_to_mel(data, sr=SAMPLE_RATE, n_mels=N_MELS, max_len=MAX_LEN):
    mel = librosa.feature.melspectrogram(y=data, sr=sr, n_mels=n_mels)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    if mel_db.shape[1] < max_len:
        pad_width = max_len - mel_db.shape[1]
        mel_db = np.pad(
            mel_db,
            pad_width=((0, 0), (0, pad_width)),
            mode="constant",
            constant_values=-80
        )
    else:
        mel_db = mel_db[:, :max_len]

    return mel_db

In [ ]:
sample_path = train_df.iloc[0]["Path"]
sample_emotion = train_df.iloc[0]["Emotions"]

data, sr = load_audio(sample_path)
mel = audio_to_mel(data, sr)

print("Sample emotion:", sample_emotion)
print("Audio shape:", data.shape)
print("Mel shape:", mel.shape)

plt.figure(figsize=(10, 4))
librosa.display.specshow(mel, sr=sr, x_axis="time", y_axis="mel")
plt.colorbar(format="%+2.0f dB")
plt.title(f"Mel Spectrogram - {sample_emotion}")
plt.show()

In [ ]:
X_train = []
y_train_text = []

for path, emotion in zip(train_df["Path"], train_df["Emotions"]):
    data, sr = load_audio(path)

    # Original
    X_train.append(audio_to_mel(data, sr))
    y_train_text.append(emotion)

    # Noise
    noisy = add_noise(data)
    X_train.append(audio_to_mel(noisy, sr))
    y_train_text.append(emotion)

    # Pitch shift
    pitched = pitch_shift(data, sr)
    X_train.append(audio_to_mel(pitched, sr))
    y_train_text.append(emotion)

    # Time stretch
    stretched = time_stretch(data)
    X_train.append(audio_to_mel(stretched, sr))
    y_train_text.append(emotion)

print("Training features:", len(X_train))
print("Training labels:", len(y_train_text))

In [ ]:
X_val = []
y_val_text = []

for path, emotion in zip(val_df["Path"], val_df["Emotions"]):
    data, sr = load_audio(path)
    X_val.append(audio_to_mel(data, sr))
    y_val_text.append(emotion)

print("Validation features:", len(X_val))
print("Validation labels:", len(y_val_text))

In [ ]:
X_test = []
y_test_text = []

for path, emotion in zip(test_df["Path"], test_df["Emotions"]):
    data, sr = load_audio(path)
    X_test.append(audio_to_mel(data, sr))
    y_test_text.append(emotion)

print("Test features:", len(X_test))
print("Test labels:", len(y_test_text))

In [ ]:
X_train = np.array(X_train, dtype=np.float32)
X_val = np.array(X_val, dtype=np.float32)
X_test = np.array(X_test, dtype=np.float32)

print("Before normalization:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train = (X_train + 80) / 80
X_val = (X_val + 80) / 80
X_test = (X_test + 80) / 80

X_train = np.clip(X_train, 0, 1)
X_val = np.clip(X_val, 0, 1)
X_test = np.clip(X_test, 0, 1)

X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("\nAfter channel dimension:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

In [ ]:
label_encoder = LabelEncoder()

y_train_int = label_encoder.fit_transform(y_train_text)
y_val_int = label_encoder.transform(y_val_text)
y_test_int = label_encoder.transform(y_test_text)

y_train = to_categorical(y_train_int, num_classes=num_classes)
y_val = to_categorical(y_val_int, num_classes=num_classes)
y_test = to_categorical(y_test_int, num_classes=num_classes)

print("Classes:", label_encoder.classes_)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

for i, name in enumerate(label_encoder.classes_):
    print(i, "->", name)

In [ ]:
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

assert X_train.ndim == 4
assert X_val.ndim == 4
assert X_test.ndim == 4
assert y_train.ndim == 2
assert y_val.ndim == 2
assert y_test.ndim == 2
assert X_train.shape[1:] == (N_MELS, MAX_LEN, 1)
assert y_train.shape[1] == num_classes

print("All checks passed.")

In [ ]:
model = Sequential()

model.add(Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=(N_MELS, MAX_LEN, 1)))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.25))

model.add(Conv2D(128, (3, 3), activation="relu", padding="same"))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.30))

model.add(Conv2D(128, (3, 3), activation="relu", padding="same"))
model.add(BatchNormalization())
model.add(MaxPooling2D((2, 2)))
model.add(Dropout(0.30))

model.add(Flatten())
model.add(Dense(128, activation="relu"))
model.add(Dropout(0.40))
model.add(Dense(num_classes, activation="softmax"))

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
rlrp = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    verbose=1,
    min_lr=1e-6
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_speech_emotion_cnn.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    batch_size=64,
    epochs=80,
    validation_data=(X_val, y_val),
    callbacks=[rlrp, early_stop, checkpoint]
)

In [ ]:
print("Best validation accuracy:", max(history.history["val_accuracy"]))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)

print("Final test loss:", test_loss)
print("Final test accuracy:", test_acc)

In [ ]:
y_pred = model.predict(X_test)

y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print(classification_report(
    y_true_classes,
    y_pred_classes,
    target_names=label_encoder.classes_
))

In [ ]:
cm = confusion_matrix(y_true_classes, y_pred_classes)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
    cmap="Blues"
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
model.save("speech_emotion_cnn_final.keras")

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

config = {
    "sample_rate": SAMPLE_RATE,
    "duration": DURATION,
    "offset": OFFSET,
    "n_mels": N_MELS,
    "max_len": MAX_LEN,
    "num_classes": int(num_classes),
    "classes": label_encoder.classes_.tolist(),
    "remove_calm": REMOVE_CALM
}

with open("config.json", "w") as f:
    json.dump(config, f, indent=4)

print("Saved:")
print("- speech_emotion_cnn_final.keras")
print("- label_encoder.pkl")
print("- config.json")

In [ ]:
def predict_emotion(audio_path):
    data, sr = load_audio(audio_path)
    mel = audio_to_mel(data, sr)

    mel = np.array(mel, dtype=np.float32)
    mel = (mel + 80) / 80
    mel = np.clip(mel, 0, 1)
    mel = mel[np.newaxis, ..., np.newaxis]

    pred = model.predict(mel)
    pred_index = np.argmax(pred, axis=1)[0]

    emotion = label_encoder.inverse_transform([pred_index])[0]
    confidence = pred[0][pred_index]

    return emotion, confidence

sample_path = test_df.iloc[0]["Path"]
actual_emotion = test_df.iloc[0]["Emotions"]

predicted_emotion, confidence = predict_emotion(sample_path)

print("Actual:", actual_emotion)
print("Predicted:", predicted_emotion)
print("Confidence:", confidence)